In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import *

apply_plot_style()
FIGURES_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
raw = pd.read_csv(PROCESSED_DIR / "heart_disease_raw.csv")
prep = pd.read_csv(PROCESSED_DIR / "heart_disease_preprocessed.csv")

print(f"raw          {raw.shape}  missing values: {int(raw.isna().sum().sum())}")
print(f"preprocessed {prep.shape}  missing values: {int(prep.isna().sum().sum())}")
print()
print("raw columns: ", ", ".join(feature_columns(raw)))
print("prep columns:", ", ".join(feature_columns(prep)))

In [ ]:
test_ids = stratified_test_ids(prep, [SITE_COL, TARGET])

# The raw arm keeps every record the source files contained. Only the shared test
# patients are held out, so it trains on the 219 records preprocessing discarded.
print(f"test patients: {len(test_ids)} (identical for both arms)")
print(f"raw  available: {len(raw)}   -> train {len(raw) - len(test_ids)}")
print(f"prep available: {len(prep)}  -> train {len(prep) - len(test_ids)}")

In [ ]:
for name, frame in [("prep", prep), ("raw", raw)]:
    mask = frame["id"].isin(test_ids)
    frame[~mask].to_csv(PROCESSED_DIR / f"{name}_train.csv", index=False)
    frame[mask].to_csv(PROCESSED_DIR / f"{name}_test.csv", index=False)
    print(f"{name}: {(~mask).sum()} train / {mask.sum()} test")

In [ ]:
parts = {n: pd.read_csv(PROCESSED_DIR / f"{n}.csv")
         for n in ["prep_train", "prep_test", "raw_train", "raw_test"]}

for n, d in parts.items():
    print(f"{n:<12} {str(d.shape):<10} positive {d[TARGET].mean():.3f}  "
          f"{dict(d[SITE_COL].value_counts())}")

print()
print("test patients identical across arms:",
      set(parts['prep_test'].id) == set(parts['raw_test'].id))
print("no train/test overlap (prep):",
      len(set(parts['prep_train'].id) & set(parts['prep_test'].id)) == 0)
print("no train/test overlap (raw): ",
      len(set(parts['raw_train'].id) & set(parts['raw_test'].id)) == 0)
extra = set(parts['raw_train'].id) - set(parts['prep_train'].id)
print(f"records the raw arm trains on that preprocessing discarded: {len(extra)}")
print("  by hospital:", pd.read_csv(PROCESSED_DIR / "raw_train.csv")
      .set_index("id").loc[sorted(extra), SITE_COL].value_counts().to_dict())